# Datamine VALIDA Process Example

This notebook demonstrates how to configure and run the **`valida`** process wrapper in `dmstudio`.

## Process Description

## VALIDA

Validate fields against each other in a Datamine file.

Commands are entered interactively to specify the nature of the required tests.

The main test commands are:

TEST FFFFFFFF.op.GGGGGGGG  |  Checks the relationship between field names **FFFFFFFF** and **GGGGGGGG**.
---|---
UNLS FFFFFFFF.op.GGGGGGGG  |  If all **UNLS** criteria are satisfied, then all **TEST** criteria are ignored for the current record.

If any test fails, the entire record is failed, and will not be written to the output file unless the **PASS** command has been given.

It is important that the described format for the **TEST** and **UNLS** commands is kept to, in particular that the **FFFFFFFF** and **GGGGGGGG** field names are entered as 8 characters, blank padded to the right if required, and the operators are 4 characters long (for example, .EQ.). There is a maximum of 20 commands permitted.

The validation commands are prompted for:

> TEST FFFFFFFF.op.GGGGGGGG

Test field **FFFFFFFF** (entered as 8 characters, blank padded if necessary) against field **GGGGGGGG**. The permissible operators .op. are .GE.,.EQ.,.LE.,.NE. If **FFFFFFFF** i s the same as **GGGGGGGG** then the test is on the same field between the previous and the current record; otherwise the test is on different fields in the same record. There may be a number of different **TEST** criteria specified. The record will only pass if it passes all TEST criteria.

> UNLS FFFFFFFF.op.GGGGGGGG

If all the given **UNLS** relationships between field **FFFFFFFF** and field **GGGGGGGG** are true, then the **TEST** criteria will not be carried out, and the record will pass. The format for and meaning of the components of the **UNLS** command are exactly the same as for the **TEST** command.

>MESS Display message on failed tests.

>PASS Write failed records to &OUT.

>FAIL Do not write failed records (default).

>LAST End of commands.

### Input Files:

* **in** (Table):
  File to be validated.
  Required=Yes

### Output Files:

* **out** (Table):
  File containing validated records.
  Required=Yes

### Fields:

### Parameters:

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('valida')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute valida
print("Running valida...")
dm_cmd.valida(
    in_i='t_assays',  # required
    out_o='t_valida_out',  # required
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("valida execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_valida_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")